# N2 — CNN-BiGRU acoustic baseline

In [ ]:
from pathlib import Path
import sys, yaml, numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
CFG=yaml.safe_load((ROOT/'config.yaml').read_text())
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
print('device:',DEVICE,'debug:',CFG['data']['debug'])

from torch.utils.data import Dataset,DataLoader
from piano_transcriber.data import MaestroTrackDataset
from piano_transcriber.acoustic import AcousticCNNBiGRU
from piano_transcriber.training import train_acoustic_epoch,evaluate_acoustic
from piano_transcriber.checkpoints import save_checkpoint

Dataset windows are cropped to fixed length. This example is an explicit training template; use a custom collate/cached window index for large full-track runs.

In [ ]:
class TrackWindows(Dataset):
 def __init__(self,base,window=512):
  self.base=base;self.items=[]
  for i in range(len(base)):
   m,y,o=base[i]
   for s in range(0,max(1,m.shape[1]-window+1),window):
    if s+window<=m.shape[1]:self.items.append((i,s,window))
 def __len__(self):return len(self.items)
 def __getitem__(self,j):
  i,s,w=self.items[j];m,y,o=self.base[i]
  return torch.tensor(m[:,s:s+w]),torch.tensor(y[s:s+w]),torch.tensor(o[s:s+w])
def dataset(split):
 return TrackWindows(MaestroTrackDataset(CFG['data']['metadata_csv'],split,
  CFG['features']['sample_rate'],CFG['features']['n_fft'],CFG['features']['hop_length'],
  CFG['features']['n_mels'],CFG['data']['debug'],CFG['data']['max_tracks'],
  CFG['features']['pitch_min'],CFG['features']['pitch_max']))
tr=DataLoader(dataset('train'),batch_size=CFG['model']['batch_size'],shuffle=True)
va=DataLoader(dataset('validation'),batch_size=CFG['model']['batch_size'])

In [ ]:
model=AcousticCNNBiGRU(CFG['features']['n_mels'],88,CFG['model']['hidden_size'],CFG['model']['gru_layers'],CFG['model']['dropout']).to(DEVICE)
opt=torch.optim.AdamW(model.parameters(),lr=CFG['model']['learning_rate']);hist={'train':[],'val':[]}
best=float('inf')
for ep in range(CFG['model']['epochs']):
 a=train_acoustic_epoch(model,tr,opt,DEVICE);b=evaluate_acoustic(model,va,DEVICE)
 hist['train'].append(a);hist['val'].append(b);print(ep+1,a,b)
 if b<best:
  best=b;save_checkpoint(ROOT/'models/acoustic.pt',model,{'hidden_size':CFG['model']['hidden_size'],'gru_layers':CFG['model']['gru_layers']},CFG['features'],{'epoch':ep+1,'val_loss':b})
plt.plot(hist['train'],label='train');plt.plot(hist['val'],label='validation');plt.legend();plt.show()

Cache sigmoid frame/onset probabilities and aligned labels per validation/test track as compressed NPZ files. Preserve clip IDs so paired statistical tests can be performed.